# FSCT 8561 — Lab 4: Network Traffic Analysis Using Scapy
**Course:** Security Applications  
**Duration:** 3 hours  
**Instructor:** Dr. Maryam R. Aliabadi

**Learning approach:** Parts 1–3 are guided, with short commands explained one at a time. In Parts 4–5, you will combine what you learned to write your own scripts. This lab uses a **recorded PCAP file**; no live capture, Npcap, or administrator privileges are required.

## Overview and learning objectives
You are analyzing recorded network traffic from a test environment. The provided file, `botnet-capture-20110812-rbot.pcap`, contains botnet-related activity. Your job is to inspect packets and look for patterns that may indicate security risks.

By the end, you should be able to:
1. Open a PCAP file using Scapy.
2. Filter TCP, TCP port 80, and UDP port 53 traffic.
3. Read IP addresses, port numbers, TCP flags, timestamps, and packet sizes.
4. Inspect readable payload data carefully.
5. Write a simple traffic summary and a threshold-based alert script.

**Ethics:** Analyze only the provided capture or traffic you are authorized to inspect. Avoid sharing sensitive payload data in screenshots or reports.

## Preparation 
Place `botnet-capture-20110812-rbot.pcap` in the same folder as this notebook. Install Scapy once if needed:

`%pip install scapy`



# Part 1 — Read and filter packets 
We will begin with one command at a time, then combine the commands.

### Step 1. Import Scapy's PCAP reader
`from scapy.all import rdpcap` means: import the function that reads packets from a saved capture file.

In [ ]:
from scapy.all import rdpcap

### Step 2. Read the PCAP file
`rdpcap()` opens the file. The variable `packets` stores the captured packets.

In [ ]:
packets = rdpcap("botnet-capture-20110812-rbot.pcap")

### Step 3. Count packets
`len()` tells us how many packets were loaded.

In [ ]:
print("Number of packets:", len(packets))

### Step 4. Look at one packet
Python starts counting at zero. `packets[0]` is the first packet. `.summary()` gives a short description.

In [ ]:
print(packets[0].summary())

### Step 5. Look at the first five packets
`packets[:5]` selects the first five. The `for` loop prints them one at a time.

In [ ]:
for pkt in packets[:5]:
    print(pkt.summary())

### Step 6. Recognize TCP
Import `TCP`. The expression `TCP in pkt` checks whether the packet contains a TCP layer. Try this on one packet first.

In [ ]:
from scapy.all import TCP

pkt = packets[0]
print("Is this TCP?", TCP in pkt)

### Step 7. Keep only TCP packets
Start with an empty list. Check packets one at a time. `.append()` adds a matching packet to the list.

In [ ]:
tcp_packets = []

for pkt in packets:
    if TCP in pkt:
        tcp_packets.append(pkt)

print("TCP packets:", len(tcp_packets))

### Step 8. Display up to 50 TCP packets
`[:50]` selects no more than 50 packets. This replaces the live-capture command `sniff(filter="tcp", count=50, ...)`.

In [ ]:
for pkt in tcp_packets[:50]:
    print(pkt.summary())

### Step 9. Filter traffic using TCP port 80
A TCP packet has a **source port** (`sport`) and a **destination port** (`dport`). Port 80 is conventionally used for HTTP. First, inspect the ports of one TCP packet.

In [ ]:
if len(tcp_packets) > 0:
    pkt = tcp_packets[0]
    print("Source port:", pkt[TCP].sport)
    print("Destination port:", pkt[TCP].dport)

Now keep TCP packets for which **either** port is 80. A port-80 match does not guarantee the payload contains HTTP.

In [ ]:
http_packets = []

for pkt in tcp_packets:
    if pkt[TCP].sport == 80 or pkt[TCP].dport == 80:
        http_packets.append(pkt)

print("TCP port 80 packets:", len(http_packets))
for pkt in http_packets[:50]:
    print(pkt.summary())

### Step 10. Filter UDP port 53 traffic
DNS commonly uses UDP port 53, although DNS can also use TCP or encrypted transports. Import `UDP` and use the same approach.

In [ ]:
from scapy.all import UDP

dns_packets = []

for pkt in packets:
    if UDP in pkt:
        if pkt[UDP].sport == 53 or pkt[UDP].dport == 53:
            dns_packets.append(pkt)

print("UDP port 53 packets:", len(dns_packets))
for pkt in dns_packets[:50]:
    print(pkt.summary())

**Checkpoint 1 — Write short answers:**
-  How many TCP, TCP-port-80, and UDP-port-53 packets did you find?
-  Why might a filter return zero packets?
-  What is the difference between filtering by protocol and filtering by port?

# Part 2 — Inspect packet headers 
We will examine **one packet first**, then repeat the same commands on a few packets.

### Step 1. Display all fields of one packet
`.show()` displays packet layers and their fields. Run this on one packet and identify Ethernet (if present), IP, and TCP/UDP.

In [ ]:
packets[0].show()

### Step 2. Identify IP packets
Import `IP` and check whether the first packet has an IP layer.

In [ ]:
from scapy.all import IP

pkt = packets[0]
print("Contains IPv4?", IP in pkt)

### Step 3. Read source and destination IP addresses
Only access `pkt[IP]` when `IP in pkt` is true.

In [ ]:
if IP in pkt:
    print("Source IP:", pkt[IP].src)
    print("Destination IP:", pkt[IP].dst)

### Step 4. Inspect the first five IPv4 packets
A loop repeats the same IP checks. The `if` statement avoids errors on non-IPv4 packets.

In [31]:
shown = 0
for pkt in packets:
    if IP in pkt:
        print(pkt[IP].src, "->", pkt[IP].dst)
        shown = shown + 1
        if shown == 5:
            break

78.40.125.4 -> 147.32.84.165
147.32.84.165 -> 78.40.125.4
147.32.84.165 -> 78.40.125.4
78.40.125.4 -> 147.32.84.165
78.40.125.4 -> 147.32.84.165


### Step 5. Read TCP flags
Flags help describe TCP communication. For example, `S` means SYN and `A` means ACK. Inspect a TCP packet.

In [ ]:
if len(tcp_packets) > 0:
    pkt = tcp_packets[0]
    print("TCP flags:", pkt[TCP].flags)

### Step 6. Read packet size and timestamp
`len(pkt)` is the packet size in bytes. `pkt.time` is the recorded timestamp. Convert it to `float` to make it easier to use in later calculations.

In [ ]:
pkt = packets[0]
print("Size (bytes):", len(pkt))
print("Timestamp:", float(pkt.time))

### Step 7. Put the fields together for five packets


In [ ]:
shown = 0
for pkt in packets:
    if IP in pkt:
        print("Source:", pkt[IP].src, "Destination:", pkt[IP].dst,
              "Bytes:", len(pkt), "Time:", float(pkt.time))
        shown = shown + 1
        if shown == 5:
            break

**Checkpoint 2 — Answer:** 
- What do source and destination IP addresses tell you? 
- What is the purpose of TCP flags? 
- Why are timestamps useful when investigating possible flooding?

# Part 3 — Inspect payloads and possible information exposure 
A packet may contain a payload (data carried above the headers). Not all payloads are readable text; many are binary or encrypted. We will inspect **small samples only**.

### Step 1. Import the Raw layer
Scapy often represents uninterpreted payload bytes using `Raw`.

In [36]:
from scapy.all import Raw

### Step 2. Check whether one packet contains Raw data
The `in` operator works here just as it did for TCP and IP.

In [ ]:
pkt = packets[0]
print("Contains Raw payload?", Raw in pkt)

### Step 3. Look at a short payload sample
Bytes are not always printable text. `repr()` shows a safe representation of the first 80 bytes without assuming the payload is readable.

In [ ]:
if Raw in pkt:
    print(repr(bytes(pkt[Raw].load)[:80]))
else:
    print("No Raw payload in this packet")

### Step 4. Find a packet that contains a payload
The loop stops at the **first** matching packet. This avoids printing large amounts of potentially sensitive data.

In [ ]:
for pkt in packets:
    if Raw in pkt:
        print("Payload sample:", repr(bytes(pkt[Raw].load)[:80]))
        break

### Step 5. Display a small readable sample
`.decode("utf-8", errors="replace")` converts bytes into text. Unreadable bytes are replaced with symbols; this does **not** decrypt encrypted traffic.

In [ ]:
for pkt in packets:
    if Raw in pkt:
        sample = bytes(pkt[Raw].load)[:80]
        print(sample.decode("utf-8", errors="replace"))
        break

### Step 6. Look for familiar HTTP header text
In **unencrypted** HTTP, payloads may include readable headers such as `User-Agent:` or `Cookie:`. The example below searches for `User-Agent:` in a small number of packets. It does not print any actual cookie or credential values.

In [ ]:
found = False
for pkt in packets[:500]:
    if Raw in pkt:
        data = bytes(pkt[Raw].load)
        if b"User-Agent:" in data:
            print("Possible plaintext HTTP header found")
            found = True
            break
if not found:
    print("No matching header found in the first 500 packets")

**Checkpoint 3 — Answer:** 
- Did you find readable payload text?
- What might plaintext HTTP expose? 
- Why can you not inspect the contents of encrypted HTTPS traffic using these simple commands?
 **Do not copy real passwords, cookies, or personal data into your report.**

# Part 4 — Your task: Build a basic traffic analyzer 
**File to create:** `Traffic_sniffer.py` 
Using only the Scapy commands and basic Python patterns from Parts 1–3, write your own script that:
1. Opens the provided PCAP.
2. Prints the source IP, destination IP, protocol (TCP or UDP), and source/destination ports for the **first 20 IPv4 TCP/UDP packets**.
3. Counts the **total TCP packets** and **total UDP packets** in the capture.
4. Prints a clear summary at the end.


**Expected format (numbers depend on the file):**
```text
Source: ... Destination: ... Protocol: TCP Source port: ... Destination port: ...
...
Total TCP packets: ...
Total UDP packets: ...
```

**Student work:** Write and test the script yourself. Do not use a completed solution from this notebook.

In [ ]:
# Part 4 — Student workspace
# Write your own traffic analysis code here, then save it as Traffic_sniffer.py.

# Part 5 — Your task: Build a simple PCAP-based IDS 
**File to create:** `Anomaly_Detector.py`

Using the same PCAP, develop a simple rule-based detector for unusually frequent packets from one source IP.

**Required rule:**
 
 More than **20 IPv4 TCP/UDP packets from the same source IP within any 5-second period**  → print one alert for that IP.


Work through these design steps:
1. Load the PCAP and inspect IPv4 TCP/UDP packets.
2. Obtain each packet's source IP using `pkt[IP].src`.
3. Obtain its timestamp using `float(pkt.time)`.
4. For each source IP, keep track of the timestamps of its recent packets. A Python dictionary can associate an IP with its list of timestamps.
5. When a new packet arrives, remove timestamps **older than five seconds** relative to the current packet, then add the current timestamp.
6. If that source has more than 20 timestamps in its current window, display an alert **only once** for that IP.
7. Print total TCP packets, total UDP packets, and the number of suspicious IPs.


**Simple test scenarios:**
- 20 packets from one IP in five seconds → no alert.
- 21 packets from one IP in five seconds → one alert.
- 21 packets spread across more than five seconds → no alert, if no five-second interval exceeds 20.
- A second burst from an already alerted IP → no duplicate alert.

**Important:** A threshold alert indicates unusually frequent traffic, **not proof of a DDoS attack**. Discuss possible false positives. The capture may or may not trigger the chosen threshold. If needed, demonstrate your logic using a small invented timestamp sequence, without modifying the original PCAP.

**Hint:** If timestamps are not already in order, sort packets by `float(pkt.time)` before applying the sliding-window rule. Keep your solution readable; advanced libraries and classes are not required.

In [ ]:
# Part 5 — Student workspace
# Write your own detection code here, then save it as Anomaly_Detector.py.

# Part 6 — Reflection questions 
Answer briefly:
1. Why is packet inspection important for network security?
2. What types of security risks can arise from unencrypted traffic?
3. What are the limitations of passive packet sniffing and PCAP analysis?
4. How does encryption help prevent information leakage?
5. Why might a simple packet-rate detector produce false positives?

# Part 7 — Security analysis and submission
Write **300–400 words** covering:
- What you observed in the provided PCAP (cite actual packet examples without revealing sensitive values).
- Potential threats and vulnerabilities suggested by the observations.
- Recommendations for reducing exposure or investigating suspicious activity.
- Limitations of packet capture and threshold-based detection.

## Deliverables
- `Traffic_sniffer.py` — your Part 4 traffic analyzer.
- `Anomaly_Detector.py` — your Part 5 IDS script.
- Screenshots or a short recording showing your code running and its output.
- One PDF report containing your reflections, 300–400-word analysis, and accessible cloud links to your code and demonstration.

**Submit one PDF:** `Lab4-FirstName-LastName-StudentNumber.pdf`.

## Grading rubric — 100 points
| Component | Points |
|---|---:|
| PCAP loading and TCP/port filtering (Part 1) | 15 |
| Correct packet header inspection (Part 2) | 15 |
| Sensitive-data / payload inspection (Part 3) | 15 |
| Traffic analysis script (Part 4) | 15 |
| PCAP-based IDS script (Part 5) | 25 |
| Reflection and security analysis (Parts 6–7) | 15 |
| **Total** | **100** |


**Good Luck**